# Unit Economics, Growth Levers & Experiment Design

This notebook extends the CRM analysis from descriptive reporting to business decision support.

### Objectives
- calculate overall and product-level unit economics
- attribute marketing spend to products using a documented heuristic
- estimate observed customer value and contribution margin
- run sensitivity scenarios for major growth levers
- identify a priority growth opportunity
- translate historical patterns into testable hypotheses
- evaluate whether proposed A/B tests are statistically feasible

> **Important:** several calculations below rely on explicit business assumptions because the CRM dataset does not contain a complete transactional ledger or product attribution for every lead. These assumptions are documented where they are used.


In [ ]:
from pathlib import Path
import sys
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import colors

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pd.options.display.max_columns = None


In [ ]:
buyers = pd.read_pickle(PROCESSED_DIR / 'buyers.pkl')
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
spend = pd.read_pickle(PROCESSED_DIR / 'spend_clean.pkl')
contacts = pd.read_pickle(PROCESSED_DIR / 'contacts_clean.pkl')
calls = pd.read_pickle(PROCESSED_DIR / 'calls_clean.pkl')

## 1. Unit Economics

In [ ]:
print('UA: ', deals['contact_name'].nunique())

In [ ]:
print('B: ', len(buyers))

In [ ]:
print(f"C1: {len(buyers) / deals['contact_name'].nunique() * 100:.2f}")

### Marketing Budget Attribution by Product

### Attribution Logic

The CRM does not provide a clean product-level advertising budget for every campaign, so this project uses a **proportional attribution heuristic**.

1. For each campaign, count confirmed buyers by product.
2. Calculate each product's share of buyers within that campaign.
3. Multiply the campaign's total marketing spend by that buyer share.
4. Sum attributed spend across campaigns.
5. Campaign spend with no confirmed buyers cannot be attributed directly. This remaining budget is distributed across products in proportion to buyer volume.

This approach keeps the full marketing budget in the model while making the attribution assumption explicit.

> **Limitation:** this is a modeling assumption, not causal multi-touch attribution. Product-level CAC should therefore be interpreted as an estimate.

In [ ]:
# Total marketing spend by campaign
spend_by_camp = spend.groupby('campaign')['spend'].sum().reset_index().rename(columns={'spend': 'total_spend'})
spend_by_camp

In [ ]:
# Product shares within campaigns based on confirmed buyers

In [ ]:
# Buyer totals by campaign
buyers_per_campaign = (buyers['campaign'].value_counts().rename('total_buyers_in_campaign').reset_index())
buyers_per_campaign

In [ ]:
# A campaign can generate buyers for multiple products
product_per_campaign = (buyers[['campaign', 'product']].value_counts().rename('buyers_count').reset_index())
product_per_campaign

In [ ]:
# Buyer counts by product within each campaign
product_per_campaign = product_per_campaign.merge(buyers_per_campaign, on='campaign', how='left')
product_per_campaign

In [ ]:
# Product share of buyers within each campaign
product_per_campaign['product_share'] = product_per_campaign['buyers_count'] / product_per_campaign['total_buyers_in_campaign']
product_per_campaign

In [ ]:
# Join campaign spend and calculate attributed spend
product_per_campaign = product_per_campaign.merge(spend_by_camp, on='campaign', how='left').fillna(0)
product_per_campaign['real_spend'] = product_per_campaign['total_spend'] * product_per_campaign['product_share']
product_per_campaign

In [ ]:
# Unattributed spend from campaigns with no confirmed buyers
total_real_budget = spend['spend'].sum()
distributed_budget = product_per_campaign['real_spend'].sum()
undistributed = total_real_budget - distributed_budget
print('Unattributed budget:', undistributed.round(2))

In [ ]:
# Campaigns in Spend that generated no confirmed buyers
campaigns_with_buyers = set(buyers['campaign'].dropna().unique())
campaigns_in_spend = set(spend['campaign'].dropna().unique())

campaigns_without_buyers = campaigns_in_spend - campaigns_with_buyers
print(f'Campaigns with no confirmed buyers: {len(campaigns_without_buyers)}')

# Spend for these campaigns equals the unattributed budget
undistributed_check = spend[spend['campaign'].isin(campaigns_without_buyers)]['spend'].sum()
print(f'Spend from campaigns with no buyers: {undistributed_check:.2f}')

# Cross-check against total minus attributed spend
print(f'Total minus attributed: {undistributed:.2f}')

In [ ]:
# Distribute unattributed budget across buyers
total_buyers = len(buyers)
extra_per_buyer = undistributed / total_buyers

print(f'Unattributed budget: {undistributed:.2f}')
print(f'Total buyers: {total_buyers}')
print(f'Additional attributed cost per buyer: {extra_per_buyer:.2f}')

In [ ]:
# Final product budget = directly attributed spend + allocated unattributed spend
product_budget = product_per_campaign.groupby('product')['real_spend'].sum().reset_index()
product_buyers_count = (buyers['product'].value_counts().reset_index(name='buyers_count'))

product_budget = product_budget.merge(product_buyers_count, on='product')
product_budget['extra_budget'] = product_budget['buyers_count'] * extra_per_buyer
product_budget['final_budget'] = product_budget['real_spend'] + product_budget['extra_budget']
product_budget['final_cac'] = (product_budget['final_budget'] / product_budget['buyers_count']).round(0)

print(product_budget)
print()
print('Sum of final product budgets:', product_budget['final_budget'].sum().round(2))
print('Total marketing spend:', total_real_budget)

In [ ]:
total_budget_digital_marketing = product_budget.loc[product_budget['product'] == 'Digital Marketing', 'final_budget'].values[0]
total_budget_ux_ui_design = product_budget.loc[product_budget['product'] == 'UX/UI Design', 'final_budget'].values[0]
total_budget_web_developer = product_budget.loc[product_budget['product'] == 'Web Developer', 'final_budget'].values[0]
total_budget_check = total_budget_digital_marketing + total_budget_ux_ui_design + total_budget_web_developer

print(f'Digital Marketing: {total_budget_digital_marketing:.2f}')
print(f'UX/UI Design: {total_budget_ux_ui_design:.2f}')
print(f'Web Developer: {total_budget_web_developer:.2f}')
print(f'Sum: {total_budget_check:.2f}')
print(f'Total marketing spend: {total_real_budget:.2f}')

### Estimated Revenue

Because the dataset does not contain a complete payment-event table, revenue is estimated from the available CRM fields.

The calculation:
- uses `initial_amount_paid` when the initial payment already covers the full offer;
- accumulates recurring-payment revenue according to months of study;
- keeps the recorded initial payment when `One Payment` conflicts with a lower initial amount;
- applies a recurring-payment assumption when payment type is missing.

The result is an **estimated observed revenue** measure rather than an audited accounting figure.

In [ ]:
# Estimate observed revenue per buyer from the CRM fields.
def calc_revenue(row):
    # If the initial payment already covers the offer, no further accumulation is needed.
    if row['initial_amount_paid'] >= row['offer_total_amount']:
        return row['initial_amount_paid']

    # Recurring payments: accumulate expected payments according to months studied.
    if row['payment_type'] == 'Recurring Payments':
        return row['initial_amount_paid'] + (
            row['offer_total_amount'] - row['initial_amount_paid']
        ) / row['course_duration'] * row['months_of_study']

    # If One Payment conflicts with a lower recorded initial payment,
    # use the observed amount rather than forcing the CRM label to be correct.
    if row['payment_type'] == 'One Payment':
        return row['initial_amount_paid']

    # If payment type is missing, use the recurring-payment pattern as an explicit assumption.
    return row['initial_amount_paid'] + (
        row['offer_total_amount'] - row['initial_amount_paid']
    ) / row['course_duration'] * row['months_of_study']


In [ ]:
buyers['revenue_per_deal'] = buyers.apply(calc_revenue, axis=1)

In [ ]:
(buyers['revenue_per_deal'].sum()).round(2)

### Estimated Payment Events per Buyer

In [ ]:
def estimated_payments_count(row) -> int:
    """Estimate the number of observed payment events for one buyer."""
    initial = row['initial_amount_paid']
    offer = row['offer_total_amount']

    if initial >= offer:
        return 1

    if row['payment_type'] == 'One Payment':
        return 1

    # A buyer cannot have more observed monthly payments than months already studied.
    return min(row['months_of_study'], row['course_duration'])


### Overall Unit Economics

In [ ]:
def unit_economics_total(
    deals: pd.DataFrame,
    buyers: pd.DataFrame,
    ac_total: float
) -> pd.DataFrame:
    """Calculate overall CRM-based unit economics."""

    ua = deals['contact_name'].nunique()                       # unique leads
    b = len(buyers)                                            # confirmed buyers
    c1 = b / ua if ua else 0.0                                 # lead -> buyer conversion

    t = buyers.apply(estimated_payments_count, axis=1).sum()   # estimated payment events
    revenue = buyers['revenue_per_deal'].sum()                 # estimated observed revenue

    aov = revenue / t if t else 0.0                            # value per payment event
    apc = t / b if b else 0.0                                  # payment events per buyer
    cltv = aov * apc                                           # estimated value per buyer
    lead_ltv = cltv * c1                                       # value per acquired lead

    ac_value = ac_total
    cpa = ac_value / ua if ua else 0.0
    cac = ac_value / b if b else 0.0

    cm_ua = ua * (cltv * c1 - cpa)
    cm_b = b * (cltv - cac)
    cltv_cac = cltv / cac if cac else np.nan

    row = pd.Series({
        'UA': ua,
        'B': b,
        'C1_%': c1 * 100,
        'T': t,
        'AOV': aov,
        'Revenue': revenue,
        'APC': apc,
        'CLTV': cltv,
        'Lead_LTV': lead_ltv,
        'AC': ac_value,
        'CPA': cpa,
        'CAC': cac,
        'CLTV_CAC': cltv_cac,
        'CM_UA': cm_ua,
        'CM_B': cm_b,
    })

    report = pd.DataFrame({'TOTAL': row}).T

    int_cols = ['UA', 'B', 'T']
    report[int_cols] = report[int_cols].round(0).astype(int)

    other_cols = [c for c in report.columns if c not in int_cols]
    report[other_cols] = report[other_cols].round(2)

    return report


### Product-Level Unit Economics

In [ ]:
def unit_economics_by_product(
    buyers: pd.DataFrame,
    ac_by_product: dict
) -> pd.DataFrame:
    """Calculate buyer-based unit economics by product.

    UA, C1, CPA, and Lead LTV are intentionally excluded at product level
    because product is not reliably available for all leads.
    """
    rows = {}

    for product, ac_value in ac_by_product.items():
        b_df = buyers[buyers['product'] == product]

        b = len(b_df)
        t = b_df.apply(estimated_payments_count, axis=1).sum()
        revenue = b_df['revenue_per_deal'].sum()

        aov = revenue / t if t else 0.0
        apc = t / b if b else 0.0
        cltv = aov * apc
        cac = ac_value / b if b else 0.0
        cm_b = b * (cltv - cac)
        cltv_cac = cltv / cac if cac else np.nan

        rows[product] = pd.Series({
            'B': b,
            'T': t,
            'AOV': aov,
            'Revenue': revenue,
            'APC': apc,
            'CLTV': cltv,
            'AC': ac_value,
            'CAC': cac,
            'CLTV_CAC': cltv_cac,
            'CM_B': cm_b,
        })

    report = pd.DataFrame(rows).T

    int_cols = ['B', 'T']
    report[int_cols] = report[int_cols].round(0).astype(int)

    other_cols = [c for c in report.columns if c not in int_cols]
    report[other_cols] = report[other_cols].round(2)

    return report


In [ ]:
# Prepare inputs and calculate unit economics

buyers['course_duration'] = buyers['course_duration'].astype(int)
buyers['revenue_per_deal'] = buyers.apply(calc_revenue, axis=1)

ac_total = total_real_budget
ac_by_product = {
    'Digital Marketing': total_budget_digital_marketing,
    'UX/UI Design': total_budget_ux_ui_design,
    'Web Developer': total_budget_web_developer,
}

table1 = unit_economics_total(deals, buyers, ac_total)
table2 = unit_economics_by_product(buyers, ac_by_product)

In [ ]:
print('=== OVERALL UNIT ECONOMICS ===')
table1

### Overall Results

- **Lead → Buyer Conversion (C1):** 4.92%
- **Customer Acquisition Cost (CAC):** approximately €179
- **Estimated Customer Lifetime Value (CLTV):** approximately €4,499
- **CLTV / CAC:** approximately **25.2×**
- **Estimated Contribution Margin:** approximately €3.6M

The very high CLTV/CAC ratio indicates substantial headroom between estimated customer value and current acquisition cost. Because CLTV is derived from CRM-based revenue assumptions, the ratio should be interpreted as a directional unit-economics indicator rather than an audited finance metric.

In [ ]:
print('=== UNIT ECONOMICS BY PRODUCT ===')
table2

In [ ]:
product_report = unit_economics_by_product(buyers, ac_by_product).reset_index()
product_report = product_report.rename(columns={'index': 'product'})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('B', 'Buyers'),
    ('CM_B', 'Contribution Margin'),
    ('CAC', 'CAC'),
    ('CLTV', 'CLTV')
]

for ax, (col, title) in zip(axes.flat, metrics):
# axes.flat iterates over the subplot array as a one-dimensional sequence
    plot_df = product_report.sort_values(col, ascending=False)

    sns.barplot(data=plot_df, x='product',y=col, color=colors['accent'], ax=ax)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=25)

    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=3)

plt.suptitle('Product Unit Economics Overview', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Product-Level Insights

**Digital Marketing**
- largest product by buyer volume and estimated contribution margin;
- highest CAC among the three main products;
- strong estimated CLTV/CAC ratio.

**UX/UI Design**
- balanced combination of scale and acquisition efficiency;
- CAC below the overall average;
- strong customer-value economics.

**Web Developer**
- lowest CAC;
- lower estimated CLTV, consistent with its shorter course duration;
- potential growth opportunities include retention, upsell, additional modules, or pricing optimization.

These comparisons use estimated product-level acquisition cost based on the attribution method described above.

## 2. Business Growth Levers

### 2.1 Baseline and Scenario Analysis

In [ ]:
# base line
base_row = table1.iloc[0]

base = {
    'UA': base_row['UA'],
    'B': base_row['B'],
    'C1': base_row['C1_%'] / 100,
    'APC': base_row['APC'],
    'AOV': base_row['AOV'],
    'AC': base_row['AC'],
    'Revenue': base_row['Revenue'],
    'CM': base_row.get('CM_UA', base_row['Revenue'] - base_row['AC'])
}

def calculate_scenario(name, ua_mult=1.0, c1_mult=1.0, apc_mult=1.0, aov_mult=1.0, ac_mult=1.0, c1_ppt=0.0):
    
    ua = base['UA'] * ua_mult
    c1 = base['C1'] * c1_mult + c1_ppt
    b = ua * c1
    apc = base['APC'] * apc_mult
    aov = base['AOV'] * aov_mult
    ac = base['AC'] * ac_mult
    
    revenue = b * apc * aov
    cm = revenue - ac
    cac = ac / b if b > 0 else 0

    impact = ((cm - base['CM']) / base['CM'] * 100) if base['CM'] != 0 else 0
    
    return {
        'Scenario': name,
        'UA': round(ua),
        'C1_%': round(c1*100, 2),
        'B': round(b),
        'APC': round(apc, 2),
        'AOV': round(aov, 2),
        'Revenue': round(revenue, 2),
        'AC': round(ac, 2),
        'CAC': round(cac, 2),
        'CM': round(cm, 2),
        'Impact_%': round(impact, 2)
    }

# Define exploratory scenarios
scenarios = []
scenarios.append(calculate_scenario('Base'))
scenarios.append(calculate_scenario('UA ↑+15% (C1 ↓, AC ↑)', ua_mult=1.15, c1_ppt=-0.003, ac_mult=1.19))
scenarios.append(calculate_scenario('C1 ↑+15%', c1_ppt=0.0073))
scenarios.append(calculate_scenario('APC ↑10%', apc_mult=1.10, ac_mult=1.10))
scenarios.append(calculate_scenario('AOV +15% (C1 ↓)', aov_mult=1.15, c1_ppt=-0.0022))

df_scenarios = pd.DataFrame(scenarios)
display(df_scenarios)

### Scenario Assumptions

Business metrics are interdependent, so the scenarios do not assume that every lever changes in isolation.

Examples of modeled side effects:
- increasing **UA** may reduce conversion slightly as targeting broadens and may increase acquisition cost;
- increasing **AOV** may reduce conversion because the purchase threshold becomes higher;
- increasing **APC** may require additional retention or upsell investment.

These side-effect values are **business assumptions used for scenario exploration**. They are not estimated statistically from the historical dataset.

### 2.2 Product Sensitivity

In [ ]:
# Baseline values used in sensitivity scenarios.
base_total = table1.loc['TOTAL']
buyers_share = buyers['product'].value_counts(normalize=True)


def calc_total_cm(
    UA,
    C1_base,
    APC,
    AOV,
    AC,
    ua_mult=1.0,
    c1_mult=1.0,
    c1_ppt=0.0,
    apc_mult=1.0,
    aov_mult=1.0,
    ac_mult=1.0
):
    ua = UA * ua_mult
    c1 = C1_base * c1_mult + c1_ppt
    b = ua * c1
    apc = APC * apc_mult
    aov = AOV * aov_mult
    ac = AC * ac_mult
    revenue = b * apc * aov
    return revenue - ac


cm_total_base = calc_total_cm(
    base_total['UA'],
    base_total['C1_%'] / 100,
    base_total['APC'],
    base_total['AOV'],
    base_total['AC']
)


In [ ]:
# Why two different product-level approaches are used:
#
# UA / C1 levers:
# Product is missing for a large share of leads, so product-level UA cannot
# be estimated reliably. These funnel-entry effects are calculated for TOTAL
# and then distributed by the current buyer mix.
#
# APC / AOV / AC levers:
# These metrics are available for confirmed buyers and can therefore be
# modeled directly at product level.


In [ ]:
# Product-level scenario engine

def run_product_scenarios(table2, buyers_share, scenarios, cm_total_base, base_total):
    results = []
    for sc in scenarios:
        if sc['type'] == 'proportional':
            new_cm_total = calc_total_cm(
                base_total['UA'], base_total['C1_%'] / 100,
                base_total['APC'], base_total['AOV'], base_total['AC'],
                ua_mult=sc.get('ua_mult', 1.0),
                c1_mult=sc.get('c1_mult', 1.0),
                c1_ppt=sc.get('c1_ppt', 0.0),
                apc_mult=sc.get('apc_mult', 1.0),
                aov_mult=sc.get('aov_mult', 1.0),
                ac_mult=sc.get('ac_mult', 1.0)
            )
            total_dif = new_cm_total - cm_total_base
            for product in table2.index:
                results.append({'scenario': sc['name'], 'product': product,
                                 'Dif': round(total_dif * buyers_share[product])})
        else:
            for product in table2.index:
                row = table2.loc[product]
                B, APC, AOV, AC = float(row['B']), float(row['APC']), float(row['AOV']), float(row['AC'])
                cm_base = B * APC * AOV - AC
                cm_new = B * (APC * sc.get('apc_mult', 1.0)) * (AOV * sc.get('aov_mult', 1.0)) - (AC * sc.get('ac_mult', 1.0))
                results.append({'scenario': sc['name'], 'product': product,
                                 'Dif': round(cm_new - cm_base)})
    return pd.DataFrame(results)


In [ ]:
# Scenario definitions

scenarios = [
    {'name': 'UA +15%',  'type': 'proportional', 'ua_mult': 1.15, 'c1_ppt': -0.003, 'ac_mult': 1.19},
    {'name': 'C1 +15%',  'type': 'proportional', 'c1_ppt': 0.0073},
    {'name': 'APC +10%', 'type': 'direct', 'apc_mult': 1.10},
    {'name': 'AOV +15%', 'type': 'direct', 'aov_mult': 1.15},
]

In [ ]:
# Calculate product-level contribution-margin deltas

df_sensitivity = run_product_scenarios(table2, buyers_share, scenarios, cm_total_base, base_total)
product_sensitivity_pivot = df_sensitivity.pivot(index='product', columns='scenario', values='Dif')

# Keep a consistent scenario order
column_order = ['UA +15%', 'C1 +15%', 'APC +10%', 'AOV +15%']
product_sensitivity_pivot = product_sensitivity_pivot[column_order]

print(product_sensitivity_pivot)

In [ ]:
base_cm = table2['CM_B'].round(0).to_dict()

result = []

for product in table2.index:
    old_cm = base_cm[product]
    row = {'Product': product, 'Baseline CM': old_cm}

    for scenario in product_sensitivity_pivot.columns:
        dif = product_sensitivity_pivot.loc[product, scenario]
        new_cm = old_cm + dif
        percent = (dif / old_cm * 100) if old_cm != 0 else 0

        row[f'{scenario} (New CM)'] = round(new_cm)
        row[f'{scenario} (%)'] = round(percent, 1)

    result.append(row)

final_table = pd.DataFrame(result).set_index('Product')

cols = ['Baseline CM']
for scenario in product_sensitivity_pivot.columns:
    cols.append(f'{scenario} (New CM)')
    cols.append(f'{scenario} (%)')

final_table = final_table[cols]

print('=== Contribution Margin Growth by Product ===')

currency_cols = ['Baseline CM'] + [
    f'{scenario} (New CM)'
    for scenario in product_sensitivity_pivot.columns
]

format_dict = {col: '{:,.0f}' for col in currency_cols}
display(final_table.style.format(format_dict))


In [ ]:
percent_df = product_sensitivity_pivot.copy().astype(float)

for scenario in percent_df.columns:
    for product in percent_df.index:
        old_cm = base_cm.get(product, 0)
        delta = product_sensitivity_pivot.loc[product, scenario]

        if old_cm != 0:
            percent_growth = (delta / old_cm) * 100
        else:
            percent_growth = 0

        percent_df.loc[product, scenario] = percent_growth


plt.figure(figsize=(13, 7))

sns.heatmap(
    percent_df.round(1),
    annot=True,
    fmt='.1f',
    cmap='Blues',
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'CM Growth (%)'}
)

plt.title('CM Growth by Product (%)', fontsize=15, pad=20)
plt.xlabel('Scenario', fontsize=12)
plt.ylabel('', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Growth-Lever Prioritization

Among the modeled levers, improving **C1 (Lead → Buyer Conversion)** produces the largest contribution-margin increase in the scenario model.

The `C1 +15%` scenario adds approximately **€559K** to estimated contribution margin without explicitly increasing acquisition budget.

This makes conversion improvement a strong candidate for deeper diagnostic analysis. The next section investigates first-response time (SLA) as one possible operational driver.

## 3. Business Metric Tree

```text
CM (Contribution Margin)
│
├── Decision Drivers
│   ├── UA   — Unique leads acquired
│   ├── C1   — Lead-to-buyer conversion
│   ├── CPA  — Cost per acquired lead
│   ├── CAC  — Customer acquisition cost
│   ├── AOV  — Average value per payment event
│   └── APC  — Average payment count per buyer
│
├── Financial Outcome
│   └── Revenue
│
├── Product / Customer Metrics
│   ├── B     — Buyers
│   ├── T     — Estimated payment events
│   ├── CLTV  — Estimated customer lifetime value
│   ├── Lead LTV — CLTV × C1
│   └── AC    — Acquisition cost
│
└── Atomic CRM Metrics
    ├── contact_id, campaign, source
    ├── spend
    ├── stage, created_time, closing_date
    ├── product
    ├── offer_total_amount, initial_amount_paid
    ├── payment_type
    ├── months_of_study, course_duration
    └── lost_reason, SLA
```

## 4. Growth Hypotheses & Experiment Design

### 4.1 First-Response Time (SLA) → Funnel Performance

In [ ]:
# Prepare deals with a recorded first-response time.
d = deals.dropna(subset=['sla_minutes']).copy()

# Unknown buyer status is treated conservatively as not purchased
# for this historical diagnostic.
d['is_buyer'] = d['is_buyer'].fillna(False)

# Equal-volume SLA quartiles.
d['sla_bucket'] = pd.qcut(d['sla_minutes'], 4, duplicates='drop')

sla_stats = d.groupby('sla_bucket', observed=False).agg(
    median_sla=('sla_minutes', 'median'),
    conversion=('is_buyer', 'mean'),
    leads=('sla_minutes', 'count')
)


In [ ]:
sla_stats['conversion_%'] = (sla_stats['conversion'] * 100).round(2)

print(sla_stats[['median_sla', 'conversion_%', 'leads']])
print()
print(
    f"Overall median SLA: {d['sla_minutes'].median():.0f} min "
    f"({d['sla_minutes'].median()/60:.1f} hours)"
)
print(
    f"Fastest-group conversion: "
    f"{sla_stats['conversion_%'].iloc[0]}%"
)
print(
    f"Slowest-group conversion: "
    f"{sla_stats['conversion_%'].iloc[-1]}%"
)
print(
    f"Ratio: "
    f"{sla_stats['conversion_%'].iloc[0] / sla_stats['conversion_%'].iloc[-1]:.2f}x"
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# SLA distribution, trimmed at the 95th percentile for readability.
axes[0].hist(
    d.loc[
        d['sla_minutes'] <= d['sla_minutes'].quantile(0.95),
        'sla_minutes'
    ],
    bins=40,
    color='#2E86F5',
    edgecolor='white'
)
axes[0].axvline(
    d['sla_minutes'].median(),
    color='red',
    linestyle='--',
    label=f"median = {d['sla_minutes'].median():.0f} min"
)
axes[0].set_title('SLA Distribution (First Response Time)')
axes[0].set_xlabel('SLA, minutes')
axes[0].set_ylabel('Number of Leads')
axes[0].legend()

# Lead-to-buyer conversion by SLA quartile.
bars = axes[1].bar(
    range(len(sla_stats)),
    sla_stats['conversion_%'],
    color=['#2E86F5', '#2ECC71', '#F4D03F', '#E63946']
)
axes[1].set_xticks(range(len(sla_stats)))
axes[1].set_xticklabels(
    [f'{int(m)} min' for m in sla_stats['median_sla']]
)
axes[1].set_title('Conversion Rate by Response Speed')
axes[1].set_xlabel('Median SLA per Group')
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].bar_label(bars, fmt='%.1f%%')

plt.tight_layout()
plt.show()


### Historical Pattern

- Median SLA across the analyzed deals is approximately **330 minutes (5.5 hours)**.
- Lead-to-buyer conversion is **6.21%** in the fastest SLA quartile versus **4.64%** in the slowest quartile.
- The quartiles contain roughly equal numbers of leads, making the comparison less sensitive to sample-size imbalance.

This is an **observational association**. It does not prove that faster response time causes higher conversion.

### Conversion by SLA Quartile

| SLA group | Median SLA | Lead → Buyer Conversion |
|---|---:|---:|
| Fastest | 27 min | **6.21%** |
| Fast | 164 min | 5.36% |
| Slow | 722 min | 5.42% |
| Slowest | 1,376 min | **4.64%** |

Overall median SLA: **330 minutes (5.5 hours)**.

### Why Final Conversion Is Difficult to Use in a Short Test

Directly testing C1 within a two-week window is problematic because the sales cycle is longer than the test window:

- median closing time for won deals is approximately **17 days**;
- only about **27.5%** of buyers close within 14 days;
- most conversions therefore mature after a two-week experiment ends.

A shorter-cycle proxy metric is needed for an operational experiment.

The metric tree therefore moves one level closer to the operational process: **Contact Success Rate**.

### Contact Success Rate

Instead of waiting for a final purchase, the experiment can measure the share of leads for whom contact was successfully established.

Unresolved stages are excluded because the dataset does not yet confirm either contact success or contact failure for those leads.

In [ ]:
# Exclude unresolved stages where contact success/failure is not yet known.
unresolved_stages = [
    'New Lead',
    'Call Delayed',
    'Need to Call - Sales',
    'Need To Call',
    'Registered on Webinar'
]

resolved = deals[
    ~deals['stage'].isin(unresolved_stages)
].dropna(subset=['sla_minutes']).copy()

# Contact failure is defined only by explicit unreachable reasons.
unreachable_reasons = [
    "Doesn't Answer",
    'Stopped Answering',
    'Invalid number'
]

resolved['contact_success'] = ~resolved['lost_reason'].isin(
    unreachable_reasons
)

resolved['sla_bucket'] = pd.qcut(
    resolved['sla_minutes'],
    4,
    duplicates='drop'
)

contact_stats = resolved.groupby(
    'sla_bucket',
    observed=False
).agg(
    median_sla=('sla_minutes', 'median'),
    contact_success=('contact_success', 'mean'),
    leads=('contact_success', 'count')
)

contact_stats['contact_success_%'] = (
    contact_stats['contact_success'] * 100
).round(2)

print(
    f"Leads with SLA: {len(d)} -> "
    f"after excluding unresolved stages: {len(resolved)}"
)
print(contact_stats[['median_sla', 'contact_success_%', 'leads']])

plt.figure(figsize=(6, 4))

ax = sns.barplot(
    x=[f'{int(m)} min' for m in contact_stats['median_sla']],
    y=contact_stats['contact_success_%'],
    color=colors['accent']
)

ax.set_title('Impact of Response Speed on Contact Success')
ax.set_xlabel('Median SLA per Group')
ax.set_ylabel('Contact Success Rate (%)')

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
print(
    'Baseline Contact Success Rate:',
    f"{resolved['contact_success'].mean():.2%}"
)


### Hypothesis

**H₀:** reducing first-response time does not change Contact Success Rate.  
**H₁:** materially reducing SLA increases Contact Success Rate.

Historical data shows a notable association between faster response and successful contact, which makes SLA a reasonable operational hypothesis to test.

### HADI Cycle

**H — Hypothesis**  
Reducing median SLA from the current level toward ≤20 minutes will increase Contact Success Rate and may subsequently improve lead-to-buyer conversion.

**A — Action**
- Test group: priority lead queue, target SLA ≤20 minutes, push notifications
- Control group: current process
- 50/50 randomization at lead level

**D — Data**
- Primary metric: Contact Success Rate
- Secondary long-term metric: C1 for the same cohort after the sales cycle matures
- Statistical framework: two-sided test of proportions, α = 0.05, power = 80%

**I — Insight**
- If the test group shows a statistically significant and practically meaningful improvement, roll out the process and continue monitoring downstream C1.
- If not, investigate other conversion bottlenecks.

### A/B Test Feasibility

In [ ]:
baseline_contact_rate = resolved['contact_success'].mean()

days_span = (
    resolved['created_time'].max()
    - resolved['created_time'].min()
).days

resolved_leads_per_day = (
    len(resolved) / days_span
    if days_span > 0
    else np.nan
)

print(f"Baseline Contact Success Rate: {baseline_contact_rate:.1%}")
print(f"Resolved leads per day: {resolved_leads_per_day:.2f}")


In [ ]:
def calculate_sample_size(p, x, power_factor=16):
    """
    Required observations per group for a two-proportion test.

    p - baseline proportion
    x - minimum detectable effect (as a fraction, e.g. 0.05 for +5 p.p.)

    Uses a simplified sample-size heuristic: n = (power_factor * p * (1-p)) / x^2,
    with power_factor=16 approximating a two-sided test at alpha=0.05, power=80%.
    """
    if x <= 0:
        return float('inf')
    n = (power_factor * p * (1 - p)) / (x ** 2)
    return math.ceil(n)


def power_analysis(p, effect_ppt, leads_per_day):
    """Print and return sample size, total size, and days needed for a given effect."""
    x = effect_ppt / 100
    n_per_group = calculate_sample_size(p, x)
    total = n_per_group * 2
    days = math.ceil(total / leads_per_day)

    print('=== Power Analysis ===')
    print(f'Baseline rate p          = {p:.1%}')
    print(f'Detectable effect        = +{effect_ppt} p.p.')
    print(f'n per group              = {n_per_group}')
    print(f'Total sample needed      = {total}')
    print(f'Recruitment time (approx) = {days} days')

    return n_per_group, total, days


In [ ]:
# Full-power design for the historically motivated +8.6 p.p. effect.
power_analysis(p=0.567, effect_ppt=8.6, leads_per_day=resolved_leads_per_day)


#### Power Calculation for an +8.6 p.p. Effect

- Baseline Contact Success Rate: 56.7%
- Minimum detectable effect: +8.6 p.p.
- Required: **532 leads per group** (**1,064 total**)
- Estimated recruitment time: **29 days**

### SMART Criteria — Full-Power Version

| Criterion | Definition |
|---|---|
| **Specific** | Reduce median SLA for new leads through priority processing |
| **Measurable** | Primary metric: Contact Success Rate (target effect +8.6 p.p.) |
| **Achievable** | Operational queue + notifications |
| **Relevant** | Higher Contact Success Rate should increase C1 and contribution margin |
| **Time-bound** | The test runs for **29 days** |

In [ ]:
# Alternative scenario: 14-day test window
power_analysis(p=0.567, effect_ppt=12.4, leads_per_day=resolved_leads_per_day)


### 14-Day Constraint

If the 14-day time limit is fixed, the test is technically possible, but only by raising the MDE to **+12.4 p.p.** (target 69.0%, n=256/group, 512 total).

**Important caveat:** this MDE is noticeably higher than the realistic effect range observed in historical data (5.4-8.6 p.p.). This means a 14-day test **cannot reliably confirm the effect** if the real uplift falls in the more likely range — the result would falsely show "no significant change" even if an improvement is present.

### Hypothesis 1 — Recommended Test Design

**Primary Metric:** Contact Success Rate (share of leads for whom contact was successfully established)  
**Minimum detectable effect (MDE):** +8.6 p.p.  
**Significance level:** α = 0.05  
**Power:** 80%  
**Sample size:** 532 leads per group (1,064 total)  
**Estimated recruitment time:** 29 days

**Guardrail Metrics:**
- overall incoming lead volume
- lead quality (share of "good" leads)

**Success criterion:**
p-value < 0.05 **and** observed uplift ≥ 8.6 p.p.

### 4.2 Demo Access → Full Purchase

In [ ]:
# Demo access is represented by a symbolic initial payment (0, 1, or 9).
demo = deals[
    deals['initial_amount_paid'].isin([0, 1, 9])
].dropna(subset=['contact_name']).copy()

buyer_contacts = set(buyers['contact_name'].dropna().unique())

unique_demo_contacts = demo['contact_name'].nunique()
unique_converted = demo.loc[
    demo['contact_name'].isin(buyer_contacts),
    'contact_name'
].nunique()

demo_conversion_rate = unique_converted / unique_demo_contacts

print(f'Unique demo contacts: {unique_demo_contacts}')
print(f'Later confirmed buyers: {unique_converted}')
print(f'Demo -> Buyer Conversion: {demo_conversion_rate:.2%}')


### Diagnostic Finding

Among **791 unique contacts** with a symbolic demo payment (0, 1, or 9), only **10** later appear among confirmed buyers.

Observed Demo → Buyer Conversion: **1.26%**.

Most conversions occur through a separate later deal, suggesting that the CRM does not show a clear, structured hand-off from demo access into the full-purchase funnel.

### Hypothesis

**H₀:** a structured demo follow-up process does not improve Demo → Buyer Conversion.  
**H₁:** a personal follow-up within 24–48 hours plus a time-limited offer increases Demo → Buyer Conversion above the observed 1.26%.

The historical data identifies the demo funnel as a weak point, but it does not establish what target uplift is realistically achievable.

### Proposed HADI Cycle

**H — Hypothesis**  
A structured, fast follow-up after demo access will improve progression to a full purchase.

**A — Action**
- Test: personal follow-up within 24–48 hours + a time-limited offer linked to the same contact
- Control: current process
- 50/50 randomization at contact level

**D — Data**
- Primary metric: Demo-to-Buyer Conversion Rate
- Because the baseline rate is very low, use an exact or appropriately modeled test rather than relying on a simple normal approximation.

**I — Insight**
- A significant uplift would justify formalizing the demo follow-up workflow.
- No uplift would shift investigation toward demo content, offer fit, or lead quality.

### Statistical Feasibility

In [ ]:
# Power calculation using the project's simplified formula (power_factor=16)
p_demo_base = 0.0126   # 1.26%
effect = 0.0874         # +8.74 p.p. (to ~10%)

demo_n_per_group = calculate_sample_size(p_demo_base, effect)
demo_total_n = demo_n_per_group * 2
demo_per_day = 791 / 353  # ~= total demo contacts / days observed

demo_days_needed = math.ceil(demo_total_n / demo_per_day)

print('=== Demo Follow-up Test Feasibility ===')
print(f'Baseline conversion : {p_demo_base:.2%}')
print(f'Target conversion   : {p_demo_base + effect:.2%}')
print(f'Per group           : {demo_n_per_group}')
print(f'Total sample        : {demo_total_n}')
print(f'Recruitment time (approx): {demo_days_needed} days')


### SMART Assessment

A demo follow-up intervention is operationally specific and measurable. At roughly **2.24 demo contacts per day**, the required sample size is small enough to fit inside a **~25-day** window.

Because the baseline conversion rate is very low (1.26%), the standard proportion-test approximation is being pushed to its limits with such small absolute counts. A Fisher's exact test is a safer choice than a z-test for evaluating the actual results, even though the sample-size planning above uses the same simplified formula as the SLA test above for consistency across this project.

### Hypothesis 2 — Feasibility Conclusion

Using the project's power-analysis formula for a baseline of **1.26% → 10%**, the test requires about **27 demo contacts per group** (**54 total**).

At the observed demo rate (~2.24/day), recruitment would take roughly **25 days**.

**Conclusion:** the demo funnel is a small-scale but statistically reachable growth opportunity — the required sample size fits inside a ~25-day test window at current traffic.

# Executive Summary: Unit Economics & Growth Opportunities

## 1. Overall Unit Economics

- **C1:** 4.92%
- **CAC:** ~€179
- **Estimated CLTV:** ~€4,499
- **CLTV/CAC:** ~25.2×
- **Estimated Contribution Margin:** ~€3.6M

The economics indicate substantial distance between estimated customer value and acquisition cost, although the revenue and product-attribution assumptions should be kept in mind.

## 2. Product Economics

| Product | Estimated Attributed Budget | Buyers | CAC | Interpretation |
|---|---:|---:|---:|---|
| Digital Marketing | €95,566 | 473 | €202 | Largest product and largest budget user |
| UX/UI Design | €38,243 | 226 | €169 | Balanced scale and acquisition efficiency |
| Web Developer | €15,715 | 137 | €115 | Lowest CAC, but lower estimated CLTV |

## 3. Priority Growth Opportunity

**Conversion efficiency** is the strongest lever in the scenario model.

Historical CRM data shows that faster response is associated with better contact and conversion outcomes, making SLA a useful operational diagnostic.

### SLA experiment
- baseline Contact Success Rate: ~56.7%
- historical fast-vs-slow gap: ~8.6 p.p.
- full-power test for that effect: 532 leads/group, ~29 days
- 14-day version: only a larger ~12.4 p.p. effect is reliably detectable (256 leads/group)

### Demo funnel
- observed Demo → Buyer Conversion: 1.26%
- potentially valuable warm audience, almost no extra acquisition cost
- required sample size: 27 demo contacts/group (54 total), reachable in ~25 days at current traffic

## 4. Business Recommendations

- **Priority 1 — SLA test (~29 days).** Fully within the company's control: needs no extra marketing budget or product change, only a reworked lead-handling process. The effect is data-backed and scales to the whole incoming lead flow.
- **Priority 2 — Demo follow-up test (~25 days).** Smaller absolute sample, but a potentially large relative gain on an already "warm" audience — worth running in parallel with, or right after, the SLA test.
- optimize Digital Marketing acquisition cost because it consumes the largest absolute budget;
- consider scaling UX/UI Design given its balance of volume and CAC;
- explore retention / upsell opportunities for Web Developer.

The main analytical takeaway is not only **which lever looks attractive**, but also **whether the available data and traffic are sufficient to test it credibly**.